# Module Imports 

### Note that project root path should be in your OS path so that you can access other modules in your project while in this jupyter notebook

In [4]:
import scipy.io as sio
import pandas as pd
import numpy as np
import os
import torch
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if project_root not in sys.path:

    sys.path.insert(0, project_root)

print("Project root:", project_root)

print(sys.path)

from data_classes.decomposition import Extract_Features
from data_classes.decomposition import Extract_Features
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
import models.classification as classify
import models.loops as loops
import models.models as models
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, Subset
import plotly.express as px
from IPython.display import display
import os

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc,
    cohen_kappa_score
)

import io
import contextlib

Project root: /Users/kanwarsingh/Desktop/AI/acoustic_signal_classification
['/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification', '/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification/evaluation', '/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification/venv/lib/python3.13/site-packages']


# Loading and Preparing Phase 2 Data 

In [12]:
# load & preprocess
P2_data = sio.loadmat('../../data/phase2_data_20220215.mat')
P2_samples = pd.DataFrame(P2_data['x'].T)
P2_labels  = pd.DataFrame(P2_data['y'].T, columns=['y'])
P2_df = pd.concat([P2_samples, P2_labels], axis=1)
bad_indices = [
    100,303,496,507,609,706,909,1011,1113,1271,1373,1475,1577,1679,1781,1883,2085,2287,
    2389,2489,2789,2891,2895,2995,3096,3497,3767,3997,4099,4201,4499,4699,5199,5301,
    5302,5500,7688,7690,7691,7692,7696,7702,7704,7708,7715,7723,7727,7810,7841,7870,
    7888,7936,7941,7965,7997,8061,8080,8081,8095,8098,8124,8125,8126,8127,8128,8132,
    8156,8157,8158,8159,8160,8161,8215,8261,8302,8310,8322,8395,8425,8477,8478,8479,
    8483,8487,8491,8496,8527,8563,8587,8642,8743,8848,
    0,94,98,102,203,403,505,607,611,708,809,1009,1111,1171,1273,1473,1575,1579,1635,
    1660,1674,1681,1881,1983,1985,2087,2088,2187,2246,2387,2589,2689,2791,2893,3196,
    3455,3495,3696,3798,3999,4199,4701,4748,4749,4798,4999,5201,5209,5244,5245,5255,
    5924,5945,5977,6128,6135,6229,6271,6299,6377,6383,6388,6389,6678,6760,6903,6906,
    6935,6936,7430,7582,7694,7698,7699,7700,7706,7710,7713,7717,7718,7777,7778,7856,
    7872,7939,7943,7960,7984,7999,8046,8048,8059,8063,8064,8068,8078,8083,8084,8085,
    8093,8165,8166,8191,8308,8366,8397,8398,8481,8485,8489,8493,8494,8498,8544,8565,
    8640,8692,8798,8902
]

P2_df = P2_df.drop(index=bad_indices, errors='ignore')
P2_df = P2_df.dropna()

shuffled_df = P2_df.sample(frac=1, random_state=42).reset_index(drop=True)
P2_df_X = shuffled_df.iloc[:, :-1]
P2_df_Y = shuffled_df.iloc[:, -1]

## Raw Features

In [13]:
P2_raw = Extract_Features(P2_df_X, P2_df_Y, feature='raw')

In [14]:
print(P2_raw.get_samples().shape, P2_raw.get_labels().shape)

(8229, 4800) (8229,)


# Temporal Features


In [15]:
FRAME_SIZE = 128
HOP_LENGTH = 32

## Amplitude Envelope

In [ ]:
P2_amp_env = Extract_Features(P2_df_X, P2_df_Y, feature="amplitude_envelope", frame_size=FRAME_SIZE, hop_length=HOP_LENGTH)

In [18]:
print(P2_amp_env.get_samples().shape, P2_amp_env.get_labels().shape)

(8229, 147) (8229,)


## Zero Crossing Rate

In [20]:
P2_zcr = Extract_Features(P2_df_X, P2_df_Y, feature="zero_crossing_rate", frame_size=FRAME_SIZE, hop_length=HOP_LENGTH)

In [ ]:
print(P2_zcr.get_samples().shape, P2_zcr.get_labels().shape)

## Root Mean Square

In [21]:
P2_rms = Extract_Features(P2_df_X, P2_df_Y, feature="rms_energy", frame_size=FRAME_SIZE, hop_length=HOP_LENGTH)

In [22]:
print(P2_rms.get_samples().shape, P2_rms.get_labels().shape)

(8229, 151) (8229,)
